# DEMO SINH VAN BAN - Nhom 08 - Hyena cho tieng Viet

Notebook nay **chi clone repo va goi module**. Khong co logic thi nghiem nao
viet trong o code, vi code trong `.ipynb` khong co test nao chay qua. Toan bo
logic nam trong `hyena_study/` va duoc phu boi `tests/test_generate.py`.

## Truoc khi chay

1. Settings -> Accelerator -> **GPU T4 x2** (hoac P100).
2. Settings -> **Internet: ON** (can cho `git clone` va tai Wikipedia).
3. Run All. Uoc tinh **25 toi 30 phut**: token hoa corpus lan dau khoang 10 phut,
   Hyena khoang 9 phut, Transformer khoang 6 phut.

## Ket qua can tai ve

O cuoi in ra danh sach tep. Tai `demo_checkpoints.zip` trong panel Output ben phai.

## Ky vong dung ve chat luong

Mo hinh 7,55 trieu tham so, perplexity khoang 51 o muc am tiet. Van ban sinh ra se
**dung dang tieng Viet nhung khong mach lac ve ngu nghia** sau vai am tiet. Day la
he qua cua quy mo, khong phai loi cai dat. Dung quang cao qua tay khi demo.


## 1. Clone repo va kiem tra moi truong


In [ ]:
import os, sys, subprocess

WORK = '/kaggle/working'
REPO_DIR = os.path.join(WORK, 'Hyena-Attention-Study')
REPO = 'https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('thu muc lam viec:', os.getcwd())

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('KHONG THAY GPU. Bat Accelerator trong Settings roi chay lai.')
print('gpu   ', torch.cuda.get_device_name(0))


## 2. Chay test truoc khi dot GPU

Bon lan chay Kaggle hong truoc day deu vi mot doan ma khong co test nao di qua.
O nay chay bo test cua duong day checkpoint. Test do thi moi train.


In [ ]:
r = subprocess.run([sys.executable, 'tests/test_generate.py'],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit('TEST THAT BAI - dung lai, khong train.')


## 3. Huan luyen Hyena, co luu checkpoint

Cau hinh **giong het thi nghiem E1** da bao cao (`vi/syllable/HHHH`, uniform,
50 trieu token, seed 0), nen mo hinh dem demo chinh la mo hinh trong bang ket qua.
Diem khac duy nhat la them co `--save_ckpt`.


In [ ]:
COMMON = [
    '--lang', 'vi', '--tokenizer', 'syllable', '--vocab_size', '16000',
    '--n_docs', '90000', '--seq_len', '512', '--batch_size', '16',
    '--lr', '3e-4', '--token_budget', '50000000',
    '--data_seed', '0', '--seed', '0',
    '--decay_mode', 'uniform',
    '--token_cache', '/kaggle/working/data_cache',
    '--out_dir', '/kaggle/working/results',
    '--save_ckpt',
]

def run_train(layers, run_name):
    cmd = [sys.executable, '-m', 'hyena_study.train',
           '--layers', layers, '--run_name', run_name] + COMMON
    print(' '.join(cmd), flush=True)
    p = subprocess.run(cmd)
    if p.returncode != 0:
        raise SystemExit('train that bai cho ' + run_name)

run_train('HHHH', 'DEMO_vi_HHHH_s0')


## 4. Huan luyen Transformer, cung ngan sach

Dung lai cache dong token vua tao o buoc 3 nen khong ton them thoi gian token hoa.
Hai mo hinh nhin thay **dung cung mot luong tin hieu**, do la dieu lam phep so sanh
tro nen hop le.


In [ ]:
run_train('AAAA', 'DEMO_vi_AAAA_s0')


## 5. Sinh thu va doi chieu voi bang ket qua

O nay goi `hyena_study.generate`, module nap lai checkpoint TU DIA. Nho vay no kiem
luon rang tep sap tai ve dung duoc that, chu khong phai chi kiem mo hinh dang nam
trong bo nho.


In [ ]:
PROMPTS = [
    'Trường Đại học Công nghệ Thông tin',
    'Việt Nam là một quốc gia nằm ở',
    'Mô hình ngôn ngữ được huấn luyện trên',
    'Hà Nội là thủ đô của',
]
with open('/kaggle/working/prompts.txt', 'w', encoding='utf-8') as fh:
    fh.write(chr(10).join(PROMPTS))

cmd = [sys.executable, '-m', 'hyena_study.generate',
       '--ckpt', '/kaggle/working/results/DEMO_vi_HHHH_s0.pt',
       '--compare', '/kaggle/working/results/DEMO_vi_AAAA_s0.pt',
       '--prompts_file', '/kaggle/working/prompts.txt',
       '--max_new_tokens', '60', '--temperature', '0.9', '--top_k', '40',
       '--n_samples', '3', '--seed', '0', '--device', 'cuda',
       '--samples_json', '/kaggle/working/results/demo_samples.json']
p = subprocess.run(cmd)
if p.returncode != 0:
    raise SystemExit('sinh van ban that bai')


## 6. Dong goi de tai ve


In [ ]:
import json, zipfile, pathlib

res = pathlib.Path('/kaggle/working/results')
wanted = sorted(p for p in res.iterdir()
                if p.name.startswith('DEMO_') or p.name == 'demo_samples.json')

zpath = pathlib.Path('/kaggle/working/demo_checkpoints.zip')
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in wanted:
        z.write(p, p.name)

print('Cac tep da dong goi:')
for p in wanted:
    print('  {:34s} {:8.2f} MB'.format(p.name, p.stat().st_size / 2**20))
print()
print('=> {}  {:.2f} MB'.format(zpath.name, zpath.stat().st_size / 2**20))

print()
print('Doi chieu voi bang E1 da bao cao (trung binh 3 seed):')
print('  Hyena 51,380  |  Transformer 63,858')
for name in ['DEMO_vi_HHHH_s0', 'DEMO_vi_AAAA_s0']:
    d = json.loads((res / (name + '.json')).read_text(encoding='utf-8'))
    print('  {}: test PPL {:.3f} | {} buoc | {:,} token'.format(
        name, d['test_ppl'], d['steps'], d['tokens_seen']))
print()
print('Lan chay nay chi la MOT seed, nen lech chut so voi trung binh 3 seed la binh thuong.')
